# TP Evaluativo Nº 1 — Proceso KDD completo: Regresión Lineal vs. Árbol de Decisión
**Minería de Datos II · ITSE · Tecnicatura Superior en Ciencia de Datos e IA**

**Dataset:** Wine Quality (Vinho Verde tinto) — UCI ML Repository (Cortez et al., 2009)
**Objetivo:** estimar el puntaje `quality` (variable continua) a partir de 11 variables fisicoquímicas, comparando una Regresión Lineal Múltiple y un `DecisionTreeRegressor`.

**Estructura (etapas KDD):**
1. Selección · 2. Preprocesamiento/Limpieza · 3. Transformación · 4. Minería de datos · 5. Interpretación/Evaluación · 6. Conocimiento/Selección del modelo · 7. Preguntas de análisis (20–24)

> Ejecutar de arriba hacia abajo (`Entorno de ejecución > Ejecutar todo`). Todas las decisiones están justificadas en las celdas de texto.

In [ ]:
# ============ Imports y configuración global ============
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42  # reproducibilidad
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110


def metricas(y_true, y_pred):
    """Devuelve MAE, RMSE y R2."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

---
# ETAPA 1 — Selección de datos

**Decisión:** se usa el dataset *Wine Quality (tinto)* porque es el indicado en la consigna: 1.599 muestras, 11 predictores numéricos continuos y un target (`quality`) que podemos tratar como variable continua para un problema de regresión.

In [ ]:
# ---- 1. Descarga y carga en un DataFrame ----
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

try:
    df_raw = pd.read_csv(URL, sep=";")          # el CSV está separado por ';'
except Exception as e:
    # Respaldo: ZIP oficial de UCI (por si el enlace directo cambia)
    print("Falló el enlace directo, probando el ZIP oficial...", e)
    import io, zipfile, urllib.request
    ZIP = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"
    with urllib.request.urlopen(ZIP) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    df_raw = pd.read_csv(z.open("winequality-red.csv"), sep=";")

df_raw.head()

In [ ]:
# ---- 2. Estructura del dataset y EDA descriptivo ----
print("Dimensiones (filas, columnas):", df_raw.shape)
print("\nTipos de datos:")
print(df_raw.dtypes)
print("\nEstadísticas descriptivas (media, desvío, mínimo, máximo, cuartiles):")
df_raw.describe().T

In [ ]:
# Distribución del target y matriz de correlación (gráficos requeridos en el informe)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x="quality", data=df_raw, ax=ax[0], color="#5b2c6f")
ax[0].set_title("Distribución de quality")
for p in ax[0].patches:
    ax[0].annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                   ha="center", va="bottom", fontsize=9)

sns.heatmap(df_raw.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0,
            ax=ax[1], annot_kws={"size": 7}, cbar=False)
ax[1].set_title("Matriz de correlación")
plt.tight_layout(); plt.show()

print("Correlación de cada variable con quality:")
print(df_raw.corr()["quality"].drop("quality").sort_values(ascending=False))

**Lectura del EDA:**
- `quality` toma valores solo entre 3 y 8, concentrados en 5 y 6 (clases extremas muy poco representadas). Esto limita la capacidad de cualquier modelo de predecir los extremos y hace que las predicciones tiendan a "regresar" hacia el centro.
- Las variables con mayor correlación con `quality` son `alcohol` (positiva) y `volatile acidity` (negativa).
- Hay correlaciones fuertes entre predictores (p. ej. `fixed acidity` con `density`, `citric acid` y `pH`) → **multicolinealidad**, relevante para interpretar los coeficientes de la regresión lineal.
- Las escalas de las variables son muy distintas (p. ej. `total sulfur dioxide` vs. `chlorides`), lo que condiciona la interpretación de coeficientes (se retoma en la Etapa 3).

---
# ETAPA 2 — Preprocesamiento y limpieza

In [ ]:
# ---- 3. Estandarizar mayúsculas/minúsculas (y nombres de columnas) ----
df = df_raw.copy()
df.columns = (df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False))

# Si hubiera columnas de texto, también se estandarizan (en este dataset no hay)
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].str.strip().str.lower()

print("Columnas estandarizadas:", list(df.columns))
print("Columnas de tipo texto:", list(df.select_dtypes(include='object').columns))

**Decisión (punto 3):** todos los nombres de columnas pasan a minúsculas, sin espacios (`fixed acidity` → `fixed_acidity`). Esto evita errores de tipeo/acceso y mantiene un formato uniforme. El dataset no contiene variables categóricas de texto, por lo que no hay valores a unificar.

In [ ]:
# ---- 4. Valores nulos ----
nulos = df.isnull().sum()
print("Nulos por columna:\n", nulos)
print("\nTotal de nulos:", int(nulos.sum()))

# Chequeo adicional: ceros que podrían ser "nulos disfrazados"
print("\nCantidad de ceros por columna (solo informativo):")
print((df == 0).sum()[lambda s: s > 0])

**Decisión (punto 4):** el dataset **no presenta valores nulos**, por lo que no se imputa ni se eliminan filas por este motivo. Los únicos ceros están en `citric_acid`, que es un valor fisicoquímicamente posible (vinos sin ácido cítrico agregado), no un dato faltante, por lo que se conservan.
*(Si hubiera habido nulos, la imputación —p. ej. mediana— debería ajustarse sólo con el conjunto de entrenamiento para no filtrar información del test.)*

In [ ]:
# ---- 5. Filas duplicadas exactas ----
n_dup = df.duplicated().sum()
print(f"Filas duplicadas exactas: {n_dup} ({n_dup/len(df):.1%} del total)")

df_con_dup = df.copy()                              # lo guardamos para la pregunta 24
df = df.drop_duplicates().reset_index(drop=True)    # eliminamos ANTES de dividir train/test
print("Tamaño tras eliminar duplicados:", df.shape)

**Decisión (punto 5):** se **eliminan** las duplicadas exactas **antes** de dividir en entrenamiento y prueba. Si se dejaran, una misma observación podría quedar en ambos conjuntos (*data leakage*): el modelo la "vería" al entrenar y luego sería evaluada en prueba, inflando artificialmente las métricas —sobre todo en modelos que memorizan, como un árbol profundo—. Ver respuesta ampliada en la pregunta 20.

In [ ]:
# ---- 6. Outliers por IQR ----
def tabla_iqr(d, target="quality"):
    filas = []
    for c in d.columns.drop(target):
        q1, q3 = d[c].quantile([.25, .75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
        n = ((d[c] < lo) | (d[c] > hi)).sum()
        filas.append({"variable": c, "Q1": q1, "Q3": q3, "lim_inf": lo, "lim_sup": hi,
                      "n_outliers": n, "% outliers": 100*n/len(d)})
    return pd.DataFrame(filas).set_index("variable").sort_values("% outliers", ascending=False)

tab_out = tabla_iqr(df)
display(tab_out)

# Qué pasaría si eliminásemos toda fila con al menos un outlier
mask_any = pd.Series(False, index=df.index)
for c in df.columns.drop("quality"):
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    mask_any |= (df[c] < q1 - 1.5*iqr) | (df[c] > q3 + 1.5*iqr)
print(f"\nFilas con al menos un outlier en alguna variable: {mask_any.sum()} ({mask_any.mean():.1%})")

fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, c in zip(axes.ravel(), df.columns.drop("quality")):
    sns.boxplot(y=df[c], ax=ax, color="#d7bde2"); ax.set_title(c)
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

**Decisión (punto 6): CONSERVAR los outliers.** Justificación:
1. **Son valores plausibles del dominio**, no errores de medición: hay vinos con mucho azúcar residual, cloruros o sulfatos altos de forma natural; no hay valores imposibles (pH fuera de 0–14, densidad absurda, negativos).
2. **Eliminarlos descartaría una fracción muy grande de datos** (ver porcentaje de filas con algún outlier) y justo los vinos "extremos" que más información aportan sobre calidad.
3. **El Árbol de Decisión es robusto a valores extremos** (parte por umbrales: un valor extremo sólo cae en una hoja). La Regresión Lineal es más sensible; por eso en la sección 7 (pregunta 21) se **verifica empíricamente** si quitar outliers del entrenamiento mejora algo.
4. Alternativa considerada: transformaciones (p. ej. `log1p` en `residual_sugar` y `chlorides`, muy asimétricas). No se aplica para mantener los coeficientes en las unidades originales y comparar en igualdad de condiciones; queda como posible mejora.

In [ ]:
# ---- 7. Unificación/estandarización de unidades y formatos ----
# Verificamos que las unidades sean coherentes y los valores plausibles
print("Tipos de dato:")
print(df.dtypes.value_counts())

checks = {
    "pH entre 0 y 14":                 df["ph"].between(0, 14).all(),
    "densidad entre 0.98 y 1.01 g/cm3": df["density"].between(0.98, 1.01).all(),
    "alcohol entre 5 y 20 %":          df["alcohol"].between(5, 20).all(),
    "SO2 libre <= SO2 total":          (df["free_sulfur_dioxide"] <= df["total_sulfur_dioxide"]).all(),
    "sin valores negativos":           (df.drop(columns=[]) >= 0).all().all(),
    "quality entero en [3, 8]":        df["quality"].between(3, 8).all() and (df["quality"] % 1 == 0).all(),
}
for k, v in checks.items():
    print(f"{'OK ' if v else 'REVISAR'} - {k}")

**Decisión (punto 7):** las unidades ya son homogéneas dentro de cada variable (g/dm³, mg/dm³, % vol., etc.) y el formato es numérico en todas las columnas; las verificaciones de plausibilidad pasan. **No se requiere conversión de unidades.** El único "cambio de escala" necesario es la estandarización estadística para la regresión lineal (Etapa 3), que es una transformación para modelar y no una unificación de unidades.

In [ ]:
# ---- 8. Tamaño final tras la limpieza ----
print(f"Dataset original : {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
print(f"Duplicadas eliminadas: {n_dup}")
print(f"Nulos tratados: 0 | Outliers eliminados: 0 (se conservan)")
print(f"Dataset final    : {df.shape[0]} filas x {df.shape[1]} columnas")

---
# ETAPA 3 — Transformación de datos

In [ ]:
# ---- 9. Separar X e y ----
X = df.drop(columns="quality")
y = df["quality"]
print("X:", X.shape, "| y:", y.shape)

# ---- 10. Partición 80/20 ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("\nDistribución de quality (proporción):")
print(pd.DataFrame({"train": y_train.value_counts(normalize=True).sort_index(),
                    "test":  y_test.value_counts(normalize=True).sort_index()}))

**Decisión (punto 10):** partición 80/20 con `random_state=42` (reproducible) y **estratificada por `quality`**: como las clases 3 y 8 son muy raras, la estratificación asegura que ambas particiones tengan la misma distribución del target y una evaluación más estable.

In [ ]:
# ---- 11. Escalado ----
# Regresión Lineal: SÍ se escala (StandardScaler ajustado SOLO con train, para no filtrar info del test)
scaler = StandardScaler().fit(X_train)
X_train_sc = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),  columns=X.columns, index=X_test.index)

# Árbol de decisión: NO se escala (usa X_train / X_test originales)
X_train_sc.describe().loc[["mean", "std"]].round(3)

**Decisión (punto 11):**
- **Regresión Lineal → se escala** con `StandardScaler` (fit sólo en train). Las predicciones y métricas de OLS no cambian con el escalado, pero los coeficientes pasan a estar en "unidades de desvío estándar" y se vuelven **comparables entre variables** (con unidades originales, un coeficiente grande puede deberse sólo a que la variable tiene una escala pequeña, p. ej. `density`).
- **Árbol de Decisión → no se escala**: elige umbrales por variable y es invariante a transformaciones monótonas; el escalado no cambia las particiones ni las importancias.

---
# ETAPA 4 — Minería de datos

In [ ]:
# ---- 12. Regresión Lineal Múltiple (sobre datos escalados) ----
lin_reg = LinearRegression().fit(X_train_sc, y_train)
print("Intercepto:", round(lin_reg.intercept_, 4))
print("Modelo de regresión lineal entrenado.")

In [ ]:
# ---- 13. Árboles de decisión de regresión: profundidades 3, 5 y sin límite ----
profundidades = [3, 5, None]
arboles = {}
for d in profundidades:
    arboles[d] = DecisionTreeRegressor(max_depth=d, random_state=SEED).fit(X_train, y_train)
    print(f"max_depth={d}: profundidad real={arboles[d].get_depth()}, hojas={arboles[d].get_n_leaves()}")

Además de las tres profundidades pedidas, se explora un rango más amplio (1 a 20) para construir la curva de R² train/test y se elige la **profundidad óptima por validación cruzada de 5 folds sobre el conjunto de entrenamiento** —**no** mirando el conjunto de prueba—, de modo que el test siga siendo una evaluación honesta.

In [ ]:
# Selección de max_depth por validación cruzada (SOLO con train)
rango_prof = list(range(1, 21))
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_r2 = {}
for d in rango_prof:
    cv_r2[d] = cross_val_score(DecisionTreeRegressor(max_depth=d, random_state=SEED),
                               X_train, y_train, cv=kf, scoring="r2").mean()

mejor_prof = max(cv_r2, key=cv_r2.get)
print(pd.Series(cv_r2, name="R2 medio CV (5 folds)").to_frame().T.round(3).to_string())
print(f"\n>>> Mejor profundidad según CV: {mejor_prof}  (R2 CV = {cv_r2[mejor_prof]:.4f})")

arbol_mejor = DecisionTreeRegressor(max_depth=mejor_prof, random_state=SEED).fit(X_train, y_train)

---
# ETAPA 5 — Interpretación y evaluación de resultados

In [ ]:
# ---- 14. Regresión Lineal: métricas en test (y train como referencia) ----
res_lr = pd.DataFrame({
    "train": metricas(y_train, lin_reg.predict(X_train_sc)),
    "test":  metricas(y_test,  lin_reg.predict(X_test_sc)),
}).T
print("REGRESIÓN LINEAL")
display(res_lr)

In [ ]:
# ---- 15. Árbol: MAE, RMSE y R2 en train y test para cada profundidad ----
filas = []
for d, modelo in arboles.items():
    mtr = metricas(y_train, modelo.predict(X_train))
    mte = metricas(y_test,  modelo.predict(X_test))
    filas.append({
        "max_depth": "sin límite" if d is None else d,
        "MAE_train": mtr["MAE"], "MAE_test": mte["MAE"],
        "RMSE_train": mtr["RMSE"], "RMSE_test": mte["RMSE"],
        "R2_train": mtr["R2"], "R2_test": mte["R2"],
        "Brecha R2 (train-test)": mtr["R2"] - mte["R2"],
    })
res_arbol = pd.DataFrame(filas).set_index("max_depth")
display(res_arbol)

In [ ]:
# Curva de R2 train/test según la profundidad (gráfico requerido)
r2_tr, r2_te = [], []
for d in rango_prof:
    m = DecisionTreeRegressor(max_depth=d, random_state=SEED).fit(X_train, y_train)
    r2_tr.append(r2_score(y_train, m.predict(X_train)))
    r2_te.append(r2_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.plot(rango_prof, r2_tr, "o-", label="R² train")
plt.plot(rango_prof, r2_te, "s-", label="R² test")
plt.axvline(mejor_prof, ls="--", color="gray", label=f"mejor profundidad CV = {mejor_prof}")
plt.axhline(0, color="black", lw=0.8)
plt.xlabel("max_depth"); plt.ylabel("R²"); plt.xticks(rango_prof)
plt.title("Árbol de decisión: R² en train y test según la profundidad")
plt.legend(); plt.tight_layout(); plt.show()

curva = pd.DataFrame({"R2_train": r2_tr, "R2_test": r2_te}, index=rango_prof)
curva["brecha"] = curva["R2_train"] - curva["R2_test"]
prof_max_test = curva["R2_test"].idxmax()
print(f"Profundidad con mayor R² de test (descriptivo): {prof_max_test}")
print(f"Profundidad elegida por CV (usada como modelo final): {mejor_prof}")
curva.round(3).T

**Análisis de sobreajuste (punto 15)** — completar mirando la tabla y el gráfico generados:
- Con profundidades bajas (1–3) el árbol es simple: R² de train y de test son parecidos y bajos → **subajuste** (sesgo alto).
- A medida que `max_depth` crece, el R² de **train sube sin parar** (hasta ≈ 1 sin límite) mientras que el R² de **test deja de mejorar y luego cae**: la **brecha se ensancha**. Ese punto de quiebre (alrededor del valor de profundidad donde el R² de test alcanza su máximo y empieza a descender) marca el inicio del **sobreajuste**: el árbol empieza a memorizar ruido de entrenamiento.
- Sin límite de profundidad, el árbol llega a una hoja por observación (R² train ≈ 1) y el R² de test queda muy por debajo del de los árboles podados: **sobreajuste extremo**.

In [ ]:
# ---- 16. Interpretación: coeficientes (LR) e importancias (árbol) ----
coef = pd.Series(lin_reg.coef_, index=X.columns).sort_values()
imp  = pd.Series(arbol_mejor.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
coef.plot.barh(ax=ax[0], color=["#c0392b" if v < 0 else "#2e86c1" for v in coef])
ax[0].set_title("Regresión Lineal — coeficientes (variables estandarizadas)")
ax[0].axvline(0, color="black", lw=0.8)
imp.plot.barh(ax=ax[1], color="#27ae60")
ax[1].set_title(f"Árbol (max_depth={mejor_prof}) — feature_importances_")
plt.tight_layout(); plt.show()

print("Top coeficientes por magnitud (LR):")
print(coef.reindex(coef.abs().sort_values(ascending=False).index).round(4).head(6))
print("\nTop importancias (árbol):")
print(imp.sort_values(ascending=False).round(4).head(6))

In [ ]:
# Vista del árbol elegido (se muestran los primeros niveles para que sea legible)
plt.figure(figsize=(20, 8))
plot_tree(arbol_mejor, feature_names=list(X.columns), filled=True, rounded=True,
          max_depth=3, fontsize=8, precision=2)
plt.title(f"Árbol de decisión (max_depth={mejor_prof}) — primeros 3 niveles")
plt.show()

**Interpretación (punto 16):** completar con los valores de las salidas anteriores. Lo esperable en este dataset es que:
- **Regresión Lineal:** `alcohol` (coeficiente positivo) y `volatile_acidity` (negativo) sean los de mayor magnitud, seguidas de `sulphates` (positivo) y `total_sulfur_dioxide` (negativo). Hay que interpretar con cautela los coeficientes de las variables correlacionadas entre sí (`fixed_acidity`, `density`, `citric_acid`, `ph`) por la **multicolinealidad**: sus signos y magnitudes pueden ser inestables.
- **Árbol:** `alcohol`, `sulphates` y `volatile_acidity` concentran la mayor parte de la importancia, ya que son las que producen las primeras particiones.
- Que ambos modelos coincidan en las variables clave refuerza la conclusión de que son los principales predictores fisicoquímicos de la calidad. Las importancias del árbol no indican el *signo* del efecto (el coeficiente lineal sí).

---
# ETAPA 6 — Selección del modelo final y conclusiones

In [ ]:
# ---- 17. Tabla comparativa final (conjunto de prueba) ----
fila_lr = metricas(y_test, lin_reg.predict(X_test_sc))
fila_ar = metricas(y_test, arbol_mejor.predict(X_test))
fila_ar_full = metricas(y_test, arboles[None].predict(X_test))

tabla_final = pd.DataFrame({
    "Regresión Lineal": fila_lr,
    f"Árbol (max_depth={mejor_prof}, elegido por CV)": fila_ar,
    "Árbol sin límite (referencia)": fila_ar_full,
}).T
display(tabla_final)

# Robustez: validación cruzada 5 folds sobre TRAIN (pipelines sin fuga de información)
cv_lr = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()),
                        X_train, y_train, cv=kf, scoring="r2")
cv_ar = cross_val_score(DecisionTreeRegressor(max_depth=mejor_prof, random_state=SEED),
                        X_train, y_train, cv=kf, scoring="r2")
robustez = pd.DataFrame({
    "R2 CV media": [cv_lr.mean(), cv_ar.mean()],
    "R2 CV desvío": [cv_lr.std(), cv_ar.std()],
    "Brecha R2 train-test": [
        res_lr.loc["train", "R2"] - res_lr.loc["test", "R2"],
        r2_score(y_train, arbol_mejor.predict(X_train)) - fila_ar["R2"]],
}, index=["Regresión Lineal", f"Árbol (depth={mejor_prof})"])
print("\nRobustez / estabilidad:")
display(robustez)

In [ ]:
# ---- 18. Selección del modelo final ----
ganador_rmse = tabla_final.iloc[:2]["RMSE"].idxmin()
ganador_r2   = tabla_final.iloc[:2]["R2"].idxmax()
print("Mejor RMSE en test :", ganador_rmse)
print("Mejor R2 en test   :", ganador_r2)

plt.figure(figsize=(6, 5))
plt.scatter(y_test, lin_reg.predict(X_test_sc), alpha=0.5, label="Reg. Lineal")
plt.scatter(y_test, arbol_mejor.predict(X_test), alpha=0.5, label=f"Árbol d={mejor_prof}")
plt.plot([3, 8], [3, 8], "k--", lw=1)
plt.xlabel("quality real"); plt.ylabel("quality predicho"); plt.legend()
plt.title("Real vs. predicho (test)"); plt.tight_layout(); plt.show()

**Criterios para la selección final (punto 18):**
| Criterio | Regresión Lineal | Árbol (podado) |
|---|---|---|
| Métricas (MAE/RMSE/R²) en test | ver tabla | ver tabla |
| Interpretabilidad | Alta: coeficiente = efecto marginal y signo de cada variable | Alta en reglas, pero con un árbol profundo se vuelve difícil de leer |
| Robustez / estabilidad | Estable (baja varianza), sensible a outliers y multicolinealidad | Alta varianza: cambia mucho con pequeñas variaciones de los datos; muy sensible a `max_depth` |
| Relaciones no lineales / interacciones | No las captura | Sí las captura |
| Sobreajuste | Bajo (brecha train-test pequeña) | Riesgo alto si la profundidad no está controlada |

**Cómo decidir:** se elige el modelo con mejor desempeño en test (RMSE/R²) **y** estabilidad en CV. Si las métricas son similares, se prefiere la **Regresión Lineal** por ser más simple, estable e interpretable (principio de parsimonia). Sólo se elegiría el árbol si su ventaja en test es clara y consistente en la validación cruzada. **Escribir aquí la decisión final con los números obtenidos en la corrida.**

### Conclusión (punto 19)
*(Redactar/ajustar con los números de la tabla final; el esquema esperable es:)*

- Con **sólo variables fisicoquímicas** se logra una capacidad predictiva **moderada**: el R² en test de ambos modelos suele ubicarse alrededor de 0,3–0,4 y el MAE en torno a 0,5 puntos de calidad. Es decir, las variables de laboratorio explican una parte —no la totalidad— de la variabilidad del puntaje.
- La calidad sensorial depende también de factores **no medidos** (variedad de uva, añada, terroir, aromas/compuestos volátiles, envejecimiento) y de la **subjetividad del panel de cata**; además `quality` es una escala discreta y concentrada en 5 y 6, lo que dificulta predecir los extremos.
- Las variables más relevantes (`alcohol`, `volatile_acidity`, `sulphates`) coinciden en ambos modelos, lo que da consistencia al hallazgo.
- **Posibles mejoras:** (1) transformar variables asimétricas (`log1p`), (2) agregar interacciones/términos polinómicos, (3) modelos de ensamble (Random Forest, Gradient Boosting) que reducen la varianza del árbol, (4) regularización (Ridge/Lasso) por la multicolinealidad, (5) plantear el problema como clasificación (calidad baja/media/alta), (6) validación cruzada repetida para una estimación más estable, (7) incorporar más variables (uva, añada, zona) y más datos en los extremos de la escala.

---
# SECCIÓN 5 — Preguntas de análisis

### Pregunta 20 — ¿Por qué eliminar duplicados exactos *antes* de dividir en train/test?
Porque si una misma observación aparece repetida y se hace el split después, **una copia puede quedar en entrenamiento y la otra en prueba**. El modelo "ya vio" ese dato, por lo que acierta sobre él sin haber generalizado: es una forma de **fuga de datos (*data leakage*)**. Consecuencias: (a) el conjunto de prueba deja de ser independiente y las métricas resultan **optimistas**; (b) se **favorece artificialmente a modelos que memorizan** (árbol profundo, k-NN) frente a los que generalizan; (c) las filas repetidas pesan más en el ajuste. Al deduplicar antes del split se garantiza que el test contenga únicamente casos que el modelo nunca vio y la evaluación refleje su desempeño real. *(Matiz: en este dataset puede haber vinos distintos con idénticas mediciones por la baja resolución de los instrumentos; aun así, la práctica conservadora es eliminarlos para evitar la fuga.)*

In [ ]:
# Evidencia empírica de la fuga: cuántas filas de test tienen un duplicado exacto en train
# usando el dataset CON duplicados (df_con_dup) y el mismo tipo de split
Xd, yd = df_con_dup.drop(columns="quality"), df_con_dup["quality"]
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.2, random_state=SEED, stratify=yd)
m = Xd_te.merge(Xd_tr.drop_duplicates(), how="left", on=list(Xd.columns), indicator=True)
pct = (m["_merge"] == "both").mean()
print(f"Con duplicados: {pct:.1%} de las filas de TEST tienen una copia exacta en TRAIN")

m2 = X_test.merge(X_train.drop_duplicates(), how="left", on=list(X.columns), indicator=True)
print(f"Sin duplicados: {(m2['_merge'] == 'both').mean():.1%} de las filas de TEST tienen una copia exacta en TRAIN")

### Pregunta 21 — ¿Eliminar o conservar los outliers detectados por IQR?
**Se conservan.** (1) Con IQR algunas variables superan el 9 % de "atípicos": eliminarlos significaría **perder muchísimas filas** (ver el porcentaje de filas con algún outlier) y reducir el tamaño muestral. (2) Son **valores plausibles del dominio** (vinos naturalmente más ricos en azúcar, cloruros o sulfatos), no errores; además, muchos corresponden a vinos de calidad extrema, justo los más difíciles de predecir y más informativos. (3) El **árbol es robusto** a extremos (los aísla en hojas con umbrales); la **regresión lineal es más sensible**, por lo que se verifica a continuación empíricamente si recortarlos en entrenamiento aporta algo. (4) Si el efecto fuese negativo, una alternativa más suave que eliminar es **transformar** (log) o usar modelos robustos (Huber).

In [ ]:
# Experimento: quitar outliers (IQR) SOLO del train y evaluar sobre el MISMO test
mask = pd.Series(True, index=X_train.index)
for c in X_train.columns:
    q1, q3 = X_train[c].quantile([.25, .75]); iqr = q3 - q1
    mask &= X_train[c].between(q1 - 1.5*iqr, q3 + 1.5*iqr)

Xtr_f, ytr_f = X_train[mask], y_train[mask]
print(f"Train original: {len(X_train)} filas | sin outliers: {len(Xtr_f)} filas "
      f"({1 - len(Xtr_f)/len(X_train):.1%} descartadas)")

sc_f = StandardScaler().fit(Xtr_f)
lr_f = LinearRegression().fit(sc_f.transform(Xtr_f), ytr_f)
ar_f = DecisionTreeRegressor(max_depth=mejor_prof, random_state=SEED).fit(Xtr_f, ytr_f)

comp_out = pd.DataFrame({
    "LR - conservando outliers":  metricas(y_test, lin_reg.predict(X_test_sc)),
    "LR - sin outliers (train)":  metricas(y_test, lr_f.predict(sc_f.transform(X_test))),
    "Árbol - conservando":        metricas(y_test, arbol_mejor.predict(X_test)),
    "Árbol - sin outliers (train)": metricas(y_test, ar_f.predict(X_test)),
}).T
display(comp_out)

Si las métricas no mejoran de forma clara al quitar outliers (lo habitual aquí), queda respaldada la decisión de **conservarlos**.

### Pregunta 22 — ¿Por qué escalar para interpretar los coeficientes de la Regresión Lineal pero no para el árbol?
- **Regresión Lineal:** el coeficiente de una variable es el cambio en `quality` por **cada unidad** de esa variable. Si las unidades/escalas son distintas (g/dm³, mg/dm³, 0–14…), los coeficientes **no son comparables**: `density` (rango ~0,01) tendrá un coeficiente enorme y `total_sulfur_dioxide` (rango ~280) uno diminuto sin que eso diga cuál importa más. Con `StandardScaler`, cada coeficiente pasa a ser el cambio en `quality` por **1 desvío estándar**, y las magnitudes se pueden comparar. El escalado **no cambia las predicciones ni las métricas** de OLS, sólo la interpretación.
- **Árbol:** cada nodo busca un umbral sobre una variable a la vez; ese umbral se acomoda a la escala de la variable, y las particiones resultantes son las mismas con o sin escalado (invarianza ante transformaciones monótonas). Por eso las importancias y las predicciones no cambian.

In [ ]:
# Demostración empírica
lr_orig = LinearRegression().fit(X_train, y_train)                 # sin escalar
comp_coef = pd.DataFrame({"coef sin escalar": lr_orig.coef_, "coef estandarizado": lin_reg.coef_}, index=X.columns)
display(comp_coef)
print("¿Mismas predicciones en test con y sin escalar? ->",
      np.allclose(lr_orig.predict(X_test), lin_reg.predict(X_test_sc)))

arbol_sc = DecisionTreeRegressor(max_depth=mejor_prof, random_state=SEED).fit(X_train_sc, y_train)
print("¿Mismas predicciones del árbol con y sin escalar? ->",
      np.allclose(arbol_sc.predict(X_test_sc), arbol_mejor.predict(X_test)))
print("¿Mismas importancias? ->", np.allclose(arbol_sc.feature_importances_, arbol_mejor.feature_importances_))

### Pregunta 23 — ¿Qué significa un R² negativo y qué revela un R² train ≈ 1 con test negativo?
$R^2 = 1 - \dfrac{SS_{res}}{SS_{tot}}$ compara el error del modelo con el de un **modelo trivial que siempre predice la media**. Un **R² negativo** indica que el modelo predice **peor que simplemente usar la media** (SS_res > SS_tot). Un árbol sin límite de profundidad crece hasta tener (casi) una hoja por observación, **memoriza** el entrenamiento y logra R² train ≈ 1 (error ≈ 0). Si en test el R² es negativo (o muy bajo) hay una brecha enorme: el árbol aprendió ruido y casos particulares en lugar de la relación general, y sus predicciones sobre datos nuevos son peores que una constante. Es la señal de **sobreajuste extremo** (varianza muy alta). Se controla limitando `max_depth`, `min_samples_leaf`, podando (`ccp_alpha`) o usando ensambles.

In [ ]:
d_full = res_arbol.loc["sin límite"]
print(f"Árbol sin límite -> R2 train = {d_full['R2_train']:.3f} | R2 test = {d_full['R2_test']:.3f} | brecha = {d_full['Brecha R2 (train-test)']:.3f}")

### Pregunta 24 — Repetir la comparación final sin haber eliminado los duplicados
Se repite **todo el pipeline** (split estratificado, escalado sólo para LR, elección de profundidad por CV, evaluación en test) con el dataset **con** duplicados y se compara contra el limpio.

In [ ]:
def correr_experimento(d, etiqueta, seed=SEED):
    Xe, ye = d.drop(columns="quality"), d["quality"]
    Xtr, Xte, ytr, yte = train_test_split(Xe, ye, test_size=0.2, random_state=seed, stratify=ye)
    sc = StandardScaler().fit(Xtr)
    lr = LinearRegression().fit(sc.transform(Xtr), ytr)

    kf_ = KFold(5, shuffle=True, random_state=seed)
    cv_ = {k: cross_val_score(DecisionTreeRegressor(max_depth=k, random_state=seed), Xtr, ytr, cv=kf_, scoring="r2").mean()
           for k in range(1, 21)}
    kbest = max(cv_, key=cv_.get)
    ar = DecisionTreeRegressor(max_depth=kbest, random_state=seed).fit(Xtr, ytr)
    ar_full = DecisionTreeRegressor(random_state=seed).fit(Xtr, ytr)

    out = pd.DataFrame({
        "Regresión Lineal": metricas(yte, lr.predict(sc.transform(Xte))),
        f"Árbol (depth={kbest} por CV)": metricas(yte, ar.predict(Xte)),
        "Árbol sin límite": metricas(yte, ar_full.predict(Xte)),
    }).T
    out.insert(0, "dataset", etiqueta)
    out.insert(1, "n_filas", len(d))
    return out

comparacion = pd.concat([
    correr_experimento(df, "SIN duplicados (limpio)"),
    correr_experimento(df_con_dup, "CON duplicados"),
])
display(comparacion)

**Análisis (a completar con la tabla anterior):**
- Con duplicados **mejoran las métricas de test, especialmente las de los modelos que memorizan** (el árbol profundo y/o sin límite), porque parte del test ya está en el entrenamiento (fuga). La Regresión Lineal casi no cambia, ya que no puede memorizar observaciones.
- Esa ventaja es **artificial**: puede **achicar la distancia entre árbol y regresión lineal o incluso invertir el ranking**, llevando a elegir un modelo "mejor" que en producción no lo es. Con datos limpios, el árbol sin límite colapsa en test y la regresión lineal (o un árbol podado) queda mejor posicionada.
- **Moraleja KDD:** las etapas tempranas (limpieza) condicionan todo lo posterior: una decisión de preprocesamiento puede cambiar el modelo seleccionado y la confianza en las métricas reportadas. Por eso se documenta y justifica cada decisión, y se hace la limpieza **antes** de particionar. Como el split es aleatorio, conviene además repetirlo con distintas semillas o usar CV repetida para saber si las diferencias son reales o ruido.

---
# ETAPA 7 — Informe (.docx) generado automáticamente con los resultados reales
Esta celda arma el informe de máx. 4 páginas (decisiones y justificación por etapa KDD, tablas, gráficos y respuestas 20–24) **usando las variables y métricas calculadas arriba**, y lo descarga. Ejecutarla al final, después de correr todo el notebook. Luego completar nombre/comisión/fecha y revisar los textos.

In [ ]:
# ============ GENERACIÓN AUTOMÁTICA DEL INFORME (.docx) con los resultados reales ============
# Ejecutar DESPUÉS de todas las celdas anteriores. Usa las variables ya calculadas en el notebook.
import subprocess, sys
try:
    import docx
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-docx"], check=True)
    import docx
from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

PURPLE = RGBColor(0x5B, 0x2A, 0x6B)
f3 = lambda v: f"{v:.3f}"
f2 = lambda v: f"{v:.2f}"

# --------------------------------------------------------------------------
# 1) FIGURAS (se guardan como PNG y se insertan en el informe)
# --------------------------------------------------------------------------
figA, axs = plt.subplots(1, 2, figsize=(10, 4.1), gridspec_kw={"width_ratios": [1, 1.5]})
cnt = df["quality"].value_counts().sort_index()
axs[0].bar(cnt.index.astype(str), cnt.values, color="#5b2c6f")
for i, v in enumerate(cnt.values):
    axs[0].text(i, v, str(v), ha="center", va="bottom", fontsize=7)
axs[0].set_title("Distribución de quality (dataset limpio)", fontsize=9)
axs[0].set_xlabel("quality", fontsize=8); axs[0].set_ylabel("frecuencia", fontsize=8)
axs[0].tick_params(labelsize=8)
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axs[1],
            annot_kws={"size": 5.5}, cbar=False, xticklabels=True, yticklabels=True)
axs[1].set_title("Matriz de correlación", fontsize=9)
axs[1].tick_params(labelsize=6)
plt.tight_layout(); figA.savefig("fig_quality_corr.png", dpi=200); plt.show()

figB, axs = plt.subplots(1, 3, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1.25, 1, 1]})
axs[0].plot(curva.index, curva["R2_train"], "o-", ms=3, label="R² train")
axs[0].plot(curva.index, curva["R2_test"], "s-", ms=3, label="R² test")
axs[0].axvline(mejor_prof, ls="--", color="gray", lw=1, label=f"d* = {mejor_prof} (CV)")
axs[0].axhline(0, color="black", lw=0.6)
axs[0].set_xlabel("max_depth", fontsize=8); axs[0].set_ylabel("R²", fontsize=8)
axs[0].set_title("Árbol: R² train/test vs. profundidad", fontsize=9)
axs[0].tick_params(labelsize=7); axs[0].legend(fontsize=7)
coef.plot.barh(ax=axs[1], color=["#c0392b" if v < 0 else "#2e86c1" for v in coef])
axs[1].set_title("Reg. Lineal: coeficientes (estandarizados)", fontsize=9)
axs[1].tick_params(labelsize=6.5); axs[1].axvline(0, color="black", lw=0.6)
imp.plot.barh(ax=axs[2], color="#27ae60")
axs[2].set_title(f"Árbol (d*={mejor_prof}): importancias", fontsize=9)
axs[2].tick_params(labelsize=6.5)
plt.tight_layout(); figB.savefig("fig_curva_coef_imp.png", dpi=200); plt.show()

# --------------------------------------------------------------------------
# 2) CIFRAS DINÁMICAS PARA EL TEXTO
# --------------------------------------------------------------------------
n_raw, n_fin = len(df_raw), len(df)
p56 = df["quality"].isin([5, 6]).mean()
cq = df.corr()["quality"].drop("quality")
v_pos, v_neg = cq.idxmax(), cq.idxmin()
r_fa_den = df["fixed_acidity"].corr(df["density"])
r_fa_ph = df["fixed_acidity"].corr(df["ph"])
out_var, out_pct = tab_out.index[0], tab_out["% outliers"].iloc[0]
pct_any = mask_any.mean()
pct_dup = n_dup / n_raw
pct_leak_con = pct
pct_leak_sin = (m2["_merge"] == "both").mean()
n_tr, n_te = len(X_train), len(X_test)

lr_tr, lr_te_m = res_lr.loc["train"], res_lr.loc["test"]
ar_tr = metricas(y_train, arbol_mejor.predict(X_train))
ar_te_m = metricas(y_test, arbol_mejor.predict(X_test))
gap_lr = lr_tr["R2"] - lr_te_m["R2"]
gap_ar = ar_tr["R2"] - ar_te_m["R2"]
full = res_arbol.loc["sin límite"]
g3, g5 = res_arbol.loc[3, "Brecha R2 (train-test)"], res_arbol.loc[5, "Brecha R2 (train-test)"]
prof_best_test = int(curva["R2_test"].idxmax())

top_coef = coef.reindex(coef.abs().sort_values(ascending=False).index).head(3)
top_imp = imp.sort_values(ascending=False).head(3)
fmt_coef = ", ".join(f"{k} ({v:+.2f})" for k, v in top_coef.items())
fmt_imp = ", ".join(f"{k} ({v:.2f})" for k, v in top_imp.items())
comunes = sorted(set(top_coef.index) & set(top_imp.index))

cv_lr_m, cv_ar_m = robustez["R2 CV media"].iloc[0], robustez["R2 CV media"].iloc[1]
cv_lr_s, cv_ar_s = robustez["R2 CV desvío"].iloc[0], robustez["R2 CV desvío"].iloc[1]
lr_te, ar_te = tabla_final.iloc[0], tabla_final.iloc[1]
elige_arbol = (ar_te["R2"] - lr_te["R2"] > 0.02) and (cv_ar_m > cv_lr_m)
modelo_final = f"Árbol de Decisión (max_depth={mejor_prof})" if elige_arbol else "Regresión Lineal Múltiple"
sel = ar_te if elige_arbol else lr_te

# --------------------------------------------------------------------------
# 3) HELPERS DE DOCUMENTO
# --------------------------------------------------------------------------
doc = Document()
sec = doc.sections[0]
sec.page_width, sec.page_height = Cm(21.0), Cm(29.7)       # A4
sec.left_margin = sec.right_margin = Cm(1.8)
sec.top_margin, sec.bottom_margin = Cm(1.6), Cm(1.5)
sec.header_distance = sec.footer_distance = Cm(0.7)
doc.styles["Normal"].font.name = "Calibri"
doc.styles["Normal"].font.size = Pt(9.5)
doc.styles["Normal"]._element.rPr.rFonts.set(qn("w:eastAsia"), "Calibri")

hp = sec.header.paragraphs[0]
hr = hp.add_run("ITSE · Tecnicatura Superior en Ciencia de Datos e IA · Minería de Datos II · TP Evaluativo Nº 1")
hr.font.size = Pt(8); hr.font.color.rgb = PURPLE
fp = sec.footer.paragraphs[0]; fp.alignment = WD_ALIGN_PARAGRAPH.CENTER
fr = fp.add_run(); fr.font.size = Pt(8)
for kind, txt in (("begin", None), ("instr", " PAGE "), ("end", None)):
    if kind == "instr":
        el = OxmlElement("w:instrText"); el.set(qn("xml:space"), "preserve"); el.text = txt
    else:
        el = OxmlElement("w:fldChar"); el.set(qn("w:fldCharType"), kind)
    fr._r.append(el)

def par(text, size=9.5, style=None, after=2, align=None, italic=False):
    """Párrafo con **negritas** inline."""
    p = doc.add_paragraph(style=style)
    for i, part in enumerate(text.split("**")):
        for j, sub in enumerate(part.split("`")):
            if sub:
                r = p.add_run(sub); r.font.size = Pt(size); r.bold = (i % 2 == 1); r.italic = italic
                if j % 2 == 1:
                    r.font.name = "Consolas"; r.font.size = Pt(size - 0.5)
    pf = p.paragraph_format
    pf.space_after, pf.space_before, pf.line_spacing = Pt(after), Pt(0), 1.0
    if align is not None:
        p.alignment = align
    return p

def bullet(text, size=9.5):
    p = par(text, size=size, style="List Bullet", after=1)
    p.paragraph_format.left_indent = Cm(0.55)
    p.paragraph_format.first_line_indent = Cm(-0.4)
    return p

def h(text):
    p = doc.add_heading(text, level=2)
    for r in p.runs:
        r.font.size = Pt(11); r.font.color.rgb = PURPLE; r.font.name = "Calibri"; r.bold = True
    pf = p.paragraph_format
    pf.space_before, pf.space_after, pf.keep_with_next = Pt(6), Pt(2), True
    return p

def set_bg(cell, hex_):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto"); shd.set(qn("w:fill"), hex_)
    tcPr.append(shd)

def tabla(header, rows, size=8, bold_row=None, col_w=None):
    t = doc.add_table(rows=1, cols=len(header))
    t.style = "Table Grid"; t.alignment = WD_TABLE_ALIGNMENT.CENTER; t.autofit = False
    def fill(cell, text, bold=False, white=False, left=False):
        cell.text = ""
        p = cell.paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.LEFT if left else WD_ALIGN_PARAGRAPH.CENTER
        p.paragraph_format.space_after = p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.keep_with_next = True
        r = p.add_run(str(text)); r.font.size = Pt(size); r.bold = bold
        if white:
            r.font.color.rgb = RGBColor(255, 255, 255)
    for i, hd in enumerate(header):
        fill(t.rows[0].cells[i], hd, bold=True, white=True, left=(i == 0)); set_bg(t.rows[0].cells[i], "5B2A6B")
    for ri, row in enumerate(rows):
        cells = t.add_row().cells
        for i, v in enumerate(row):
            fill(cells[i], v, bold=(bold_row == ri), left=(i == 0))
            if bold_row == ri:
                set_bg(cells[i], "EBDEF0")
    if col_w:
        for i, w in enumerate(col_w):
            t.columns[i].width = Cm(w)
        for row in t.rows:
            for i, w in enumerate(col_w):
                row.cells[i].width = Cm(w)
    doc.add_paragraph().paragraph_format.space_after = Pt(0)
    doc.paragraphs[-1].runs and None
    return t

def figura(path, width_cm, caption):
    p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_after = Pt(0); p.paragraph_format.keep_with_next = True
    p.add_run().add_picture(path, width=Cm(width_cm))
    par(caption, size=8, align=WD_ALIGN_PARAGRAPH.CENTER, italic=True, after=3)

# --------------------------------------------------------------------------
# 4) CONTENIDO DEL INFORME
# --------------------------------------------------------------------------
t = doc.add_paragraph(); t.alignment = WD_ALIGN_PARAGRAPH.CENTER; t.paragraph_format.space_after = Pt(0)
r = t.add_run("Proceso KDD con Wine Quality (Vinho Verde tinto): Regresión Lineal vs. Árbol de Decisión")
r.bold = True; r.font.size = Pt(14); r.font.color.rgb = PURPLE
par("Alumno/a: ______________________   Comisión: ________   Fecha: ____/____/2026",
    size=8.5, align=WD_ALIGN_PARAGRAPH.CENTER, after=4)
par("**Objetivo:** estimar el puntaje de calidad (`quality`, variable continua) de un vino a partir de sus "
    "propiedades fisicoquímicas, comparando una Regresión Lineal Múltiple y un Árbol de Decisión de Regresión, "
    "y documentando y justificando cada decisión del proceso KDD.")

h("1. Selección")
par(f"Se usa el dataset *Wine Quality (tinto)* de UCI (Cortez et al., 2009): **{n_raw:,} muestras**, 11 predictores "
    f"fisicoquímicos continuos y `quality` (mediana de ≥3 catadores, observado 3–8). Es el indicado para el problema "
    f"y su target numérico permite plantear regresión. Se cargó desde el enlace oficial con `pd.read_csv(sep=';')`. "
    f"**EDA (Fig. 1):** `quality` se concentra en 5 y 6 ({p56:.0%} de las muestras) y las clases extremas son raras, lo que "
    f"dificulta predecir extremos. Las mayores correlaciones con el target son `{v_pos}` (r={cq[v_pos]:+.2f}) y "
    f"`{v_neg}` (r={cq[v_neg]:+.2f}). Hay multicolinealidad entre predictores (p. ej. `fixed_acidity`–`density` r={r_fa_den:.2f}; "
    f"`fixed_acidity`–`ph` r={r_fa_ph:.2f}), relevante para interpretar coeficientes. Las escalas difieren mucho entre variables.".replace("*Wine Quality (tinto)*", "Wine Quality (tinto)"))
figura("fig_quality_corr.png", 16.8, "Figura 1. Distribución de quality y matriz de correlación.")

h("2. Preprocesamiento y limpieza")
bullet(f"**Formato:** nombres de columnas en minúsculas y sin espacios (`fixed acidity` → `fixed_acidity`); no hay variables de texto.")
bullet(f"**Nulos:** {int(df_raw.isnull().sum().sum())} → no se imputa ni se descartan filas. Los ceros de `citric_acid` son valores posibles del dominio, no faltantes.")
bullet(f"**Duplicados exactos:** {n_dup} filas ({pct_dup:.1%}). Se **eliminan antes del split** para evitar que una misma muestra quede en train y test (fuga de datos).")
bullet(f"**Outliers (IQR):** hasta {out_pct:.1f}% en `{out_var}` y {pct_any:.0%} de las filas con al menos uno. Se **conservan**: son valores plausibles "
       f"(no errores), eliminarlos descartaría gran parte de los datos y a los vinos extremos más informativos, y el árbol es robusto a extremos. "
       f"Se verificó empíricamente el efecto sobre la regresión lineal (pregunta 21).")
bullet("**Unidades/formatos:** homogéneos en cada variable (g/dm³, mg/dm³, % vol.); se verificó plausibilidad (pH en 0–14, densidad ≈1, SO₂ libre ≤ total, sin negativos). No se requiere conversión.")
bullet(f"**Tamaño final:** {n_raw:,} × {df_raw.shape[1]} → **{n_fin:,} × {df.shape[1]}**.")

h("3. Transformación")
par(f"Se separan X (11 predictores) e y (`quality`) y se divide 80/20 con `random_state=42`, **estratificando por `quality`** (las clases 3 y 8 son raras) → "
    f"**train = {n_tr:,}** y **test = {n_te:,}**. **Escalado:** `StandardScaler` ajustado sólo con train para la **Regresión Lineal** (hace comparables los coeficientes; "
    f"no altera sus predicciones). **No se escala el árbol**: elige umbrales por variable y es invariante a transformaciones monótonas.")

h("4. Minería de datos")
par(f"Se entrenó una Regresión Lineal Múltiple (sobre datos escalados) y árboles `DecisionTreeRegressor` con `max_depth` = 3, 5 y sin límite, "
    f"explorando además 1–20 para la curva. La profundidad final **d\\* = {mejor_prof}** se eligió por **validación cruzada de 5 folds sobre train** "
    f"(R² CV = {cv_r2[mejor_prof]:.3f}), no mirando el test, para mantenerlo como evaluación honesta.".replace("d\\*", "d*"))

h("5. Interpretación y evaluación")
filas5 = [["Regresión Lineal", f3(lr_tr["MAE"]), f3(lr_te_m["MAE"]), f3(lr_tr["RMSE"]), f3(lr_te_m["RMSE"]),
           f3(lr_tr["R2"]), f3(lr_te_m["R2"]), f3(gap_lr)]]
for k in res_arbol.index:
    rr = res_arbol.loc[k]
    lab = f"Árbol (max_depth={k})" + (" ★ CV" if k == mejor_prof else "")
    filas5.append([lab, f3(rr["MAE_train"]), f3(rr["MAE_test"]), f3(rr["RMSE_train"]), f3(rr["RMSE_test"]),
                   f3(rr["R2_train"]), f3(rr["R2_test"]), f3(rr["Brecha R2 (train-test)"])])
if mejor_prof not in [3, 5]:
    filas5.append([f"Árbol (max_depth={mejor_prof}) ★ CV", f3(ar_tr["MAE"]), f3(ar_te_m["MAE"]), f3(ar_tr["RMSE"]),
                   f3(ar_te_m["RMSE"]), f3(ar_tr["R2"]), f3(ar_te_m["R2"]), f3(gap_ar)])
tabla(["Modelo", "MAE train", "MAE test", "RMSE train", "RMSE test", "R² train", "R² test", "Brecha R²"], filas5,
      col_w=[4.3, 1.8, 1.8, 1.9, 1.9, 1.8, 1.8, 1.9])
par("Tabla 1. Métricas en entrenamiento y prueba (★ = profundidad elegida por validación cruzada).", size=8,
    align=WD_ALIGN_PARAGRAPH.CENTER, italic=True, after=3)
par(f"**Sobreajuste.** La brecha de R² (train–test) pasa de {g3:.3f} con profundidad 3 a {g5:.3f} con 5 y a **{full['Brecha R2 (train-test)']:.3f} sin límite** "
    f"(R² train = {full['R2_train']:.3f}; R² test = {full['R2_test']:.3f}). En la Fig. 2, el R² de test alcanza su máximo cerca de la profundidad {prof_best_test} y a partir de ahí "
    f"deja de mejorar mientras el de train sigue subiendo: ahí **empieza el sobreajuste**. Profundidades muy bajas producen subajuste. La regresión lineal tiene una brecha mínima ({gap_lr:.3f}).")
figura("fig_curva_coef_imp.png", 17.2, "Figura 2. Curva de R² train/test según profundidad, coeficientes de la regresión e importancias del árbol.")
par(f"**Variables relevantes.** Regresión Lineal (mayor |coef| estandarizado): {fmt_coef}. Árbol: {fmt_imp}. "
    + (f"Coinciden en {', '.join('`'+c+'`' for c in comunes)}, lo que refuerza la conclusión. " if comunes else "")
    + "Los coeficientes de variables correlacionadas entre sí (acidez fija, densidad, pH) deben leerse con cautela por la multicolinealidad; "
      "las importancias del árbol no indican el signo del efecto.")

h("6. Conocimiento: selección del modelo y conclusiones")
filas6 = [
    ["Regresión Lineal", f3(lr_te["MAE"]), f3(lr_te["RMSE"]), f3(lr_te["R2"]), f"{cv_lr_m:.3f} ± {cv_lr_s:.3f}"],
    [f"Árbol (max_depth={mejor_prof})", f3(ar_te["MAE"]), f3(ar_te["RMSE"]), f3(ar_te["R2"]), f"{cv_ar_m:.3f} ± {cv_ar_s:.3f}"],
]
tabla(["Modelo", "MAE test", "RMSE test", "R² test", "R² CV (5 folds, train)"], filas6,
      bold_row=(1 if elige_arbol else 0), col_w=[5.0, 2.4, 2.4, 2.4, 4.0])
par("Tabla 2. Comparación final en el conjunto de prueba (fila resaltada = modelo elegido).", size=8,
    align=WD_ALIGN_PARAGRAPH.CENTER, italic=True, after=3)
if elige_arbol:
    just = (f"El árbol supera a la regresión en R² de test ({ar_te['R2']:.3f} vs. {lr_te['R2']:.3f}) y en validación cruzada ({cv_ar_m:.3f} vs. {cv_lr_m:.3f}); "
            f"la mejora es consistente, por lo que se acepta su mayor varianza y se mantiene podado para controlar el sobreajuste.")
else:
    just = (f"El árbol no supera de forma clara y consistente a la regresión (R² test {ar_te['R2']:.3f} vs. {lr_te['R2']:.3f}; R² CV {cv_ar_m:.3f} vs. {cv_lr_m:.3f}). "
            f"Con métricas similares se prefiere la regresión por **parsimonia**: es más simple, **interpretable** (cada coeficiente da signo y magnitud del efecto), "
            f"**más estable** (desvío en CV {cv_lr_s:.3f} vs. {cv_ar_s:.3f}) y con brecha train–test mínima ({gap_lr:.3f} vs. {gap_ar:.3f}).")
par(f"**Modelo final: {modelo_final}.** (Regla: se elige el árbol sólo si mejora el R² de test en más de 0,02 y también la CV; si no, se prefiere el modelo más simple.) " + just)
par(f"**Conclusión.** El modelo final logra R² = {sel['R2']:.3f}, MAE = {sel['MAE']:.3f} y RMSE = {sel['RMSE']:.3f}: las variables fisicoquímicas explican "
    f"aproximadamente el {max(sel['R2'],0):.0%} de la variabilidad de `quality`, es decir, **capacidad predictiva moderada**. El resto se debe a factores no medidos "
    f"(variedad de uva, añada, terroir, aromas), a la subjetividad del panel de cata y a que el target es discreto y concentrado en 5–6. "
    f"**Mejoras posibles:** transformaciones (log) de variables asimétricas, interacciones, regularización (Ridge/Lasso), ensambles (Random Forest, Gradient Boosting), "
    f"plantear el problema como clasificación (baja/media/alta), validación cruzada repetida y más variables/datos en los extremos.")

h("7. Respuestas a las preguntas de análisis")
par(f"**P20 – Duplicados antes del split.** Si se dividen con duplicados, una misma muestra puede caer en train y en test (data leakage): el test deja de ser independiente, "
    f"las métricas se inflan y se favorece a los modelos que memorizan. Aquí había {n_dup} duplicados ({pct_dup:.1%}) y, con ellos, el {pct_leak_con:.1%} de las filas de test tenía una copia "
    f"exacta en train (vs. {pct_leak_sin:.1%} tras deduplicar).")
a_lr, b_lr = comp_out.loc["LR - conservando outliers", "R2"], comp_out.loc["LR - sin outliers (train)", "R2"]
a_ar, b_ar = comp_out.loc["Árbol - conservando", "R2"], comp_out.loc["Árbol - sin outliers (train)", "R2"]
par(f"**P21 – Outliers.** Se conservan: son valores plausibles del dominio, quitarlos eliminaría {pct_any:.0%} de las filas (y a los vinos extremos), y el árbol es robusto a extremos "
    f"(la regresión lineal es más sensible). Verificación: quitarlos sólo del train cambia el R² de test de {a_lr:.3f} a {b_lr:.3f} en la regresión lineal y de {a_ar:.3f} a {b_ar:.3f} en el árbol; "
    + ("la mejora no justifica descartar datos." if max(b_lr - a_lr, b_ar - a_ar) < 0.02 else "aunque hay cierta mejora, no compensa la pérdida de información; una alternativa es transformar (log) o usar modelos robustos."))
ok_lr = np.allclose(lr_orig.predict(X_test), lin_reg.predict(X_test_sc))
ok_ar = np.allclose(arbol_sc.predict(X_test_sc), arbol_mejor.predict(X_test)) and np.allclose(arbol_sc.feature_importances_, arbol_mejor.feature_importances_)
par(f"**P22 – Escalado.** El coeficiente es el cambio en `quality` por unidad de cada variable; con unidades dispares (`density` varía ≈0,01 y `total_sulfur_dioxide` ≈280) no son comparables. "
    f"Estandarizadas, cada coeficiente es el cambio por 1 desvío y sus magnitudes se comparan (las predicciones no cambian: {'verificado' if ok_lr else 'revisar'}). "
    f"El árbol decide por umbrales sobre una variable a la vez, invariantes a transformaciones monótonas: mismas particiones, predicciones e importancias con o sin escalar ({'verificado' if ok_ar else 'revisar'}).")
if full["R2_test"] < 0:
    p23 = (f"Un **R² negativo** indica que el modelo predice **peor que la media** (SS_res > SS_tot). Con R² train = {full['R2_train']:.3f} y test = {full['R2_test']:.3f}, el árbol sin límite "
           f"memorizó el entrenamiento (una hoja por observación) y no generaliza: **sobreajuste extremo** (varianza muy alta).")
else:
    p23 = (f"Un **R² negativo** indicaría que el modelo predice **peor que la media** (SS_res > SS_tot). En esta corrida el árbol sin límite obtuvo R² train = {full['R2_train']:.3f} y test = {full['R2_test']:.3f} "
           f"(no negativo, pero con brecha de {full['Brecha R2 (train-test)']:.3f}): memoriza el entrenamiento (una hoja por observación) y generaliza mal, evidencia de **sobreajuste extremo**. Se controla limitando `max_depth`/`min_samples_leaf`, podando o con ensambles.")
par("**P23 – R² negativo.** " + p23)

def g24(ds, key):
    sub = comparacion[comparacion["dataset"] == ds]
    return sub[sub.index.str.startswith(key)].iloc[0]
L, D = "SIN duplicados (limpio)", "CON duplicados"
filas24 = []
for lab, key in [("Regresión Lineal", "Regresión"), ("Árbol podado (CV)", "Árbol ("), ("Árbol sin límite", "Árbol sin")]:
    a, b = g24(L, key), g24(D, key)
    filas24.append([lab, f3(a["RMSE"]), f3(b["RMSE"]), f3(a["R2"]), f3(b["R2"])])
win_L = "Regresión Lineal" if g24(L, "Regresión")["R2"] >= g24(L, "Árbol (")["R2"] else "Árbol podado"
win_D = "Regresión Lineal" if g24(D, "Regresión")["R2"] >= g24(D, "Árbol (")["R2"] else "Árbol podado"
full_L, full_D = g24(L, "Árbol sin")["R2"], g24(D, "Árbol sin")["R2"]
lr_L, lr_D = g24(L, "Regresión")["R2"], g24(D, "Regresión")["R2"]
par(f"**P24 – Sin eliminar duplicados** (pipeline completo repetido; Tabla 3).", after=2)
tabla(["Modelo (test)", "RMSE limpio", "RMSE con dup.", "R² limpio", "R² con dup."], filas24, col_w=[4.6, 2.6, 2.8, 2.6, 2.8])
par("Tabla 3. Comparación con y sin eliminar duplicados.", size=8, align=WD_ALIGN_PARAGRAPH.CENTER, italic=True, after=3)
par(f"El mejor modelo por R² de test es **{win_L}** con datos limpios y **{win_D}** con duplicados: "
    + ("la conclusión sobre qué modelo es mejor **no cambia**" if win_L == win_D else "la conclusión **cambia**")
    + f". El R² del árbol sin límite pasa de {full_L:.3f} a {full_D:.3f} y el de la regresión de {lr_L:.3f} a {lr_D:.3f}"
    + (": los modelos que memorizan son los más beneficiados por la fuga, una ventaja artificial que no se sostendría en producción." if full_D > full_L
       else ": en esta partición el efecto no fue visible, aunque el riesgo de fuga persiste.")
    + " **Moraleja KDD:** una decisión temprana de limpieza puede alterar las métricas, el ranking de modelos y la confianza en las conclusiones; "
      "por eso se documenta y se hace antes de particionar. Conviene repetir con varias semillas o CV repetida para distinguir diferencias reales de ruido.")

OUT = "Informe_TP1_KDD_Vino.docx"
doc.save(OUT)
print(f"Informe generado: {OUT}")
print("⚠ Verificá que no supere las 4 páginas (consigna). Completá nombre/comisión/fecha y revisá los textos con tus resultados.")
try:
    from google.colab import files
    files.download(OUT)
except Exception:
    pass